# Phase 6: Feature Engineering (ML target)

Builds the "high-value transaction" binary target (top 20% by value) and the feature set used by the classification models.

## Setup

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'],
                  dtype={'debtor_code': str, 'product_code': str})

# Keep genuine sales transactions (positive value) -- excludes returns, till-rounding, zero-value lines
df = df[df['value_zar'] > 0].copy()

## Target: high-value transaction = top 20% by value (80th percentile threshold)

In [3]:
threshold = df['value_zar'].quantile(0.80)
df['high_value'] = (df['value_zar'] >= threshold).astype(int)

print(f"High-value threshold (80th pct): R{threshold:.2f}")
print(df['high_value'].value_counts(normalize=True))

High-value threshold (80th pct): R221.71
high_value
0    0.799401
1    0.200599
Name: proportion, dtype: float64


## Date-derived features

In [4]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

## Reduce product_code cardinality: keep top 40 most frequent, bucket rest as 'OTHER'

In [5]:
top_products = df['product_code'].value_counts().nlargest(40).index
df['product_code_grp'] = np.where(df['product_code'].isin(top_products), df['product_code'], 'OTHER')

# Feature set (excludes value_zar, avg_price_per_kg, product_desc -- leakage / redundant)
feature_cols = ['qty', 'mass_kg', 'year', 'month', 'day', 'day_of_week', 'quarter',
                 'is_weekend', 'debtor_code', 'product_code_grp', 'doc_type']
target_col = 'high_value'

model_df = df[feature_cols + [target_col, 'date', 'value_zar']].copy()
model_df.to_parquet('../data/model_ready.parquet', index=False)
print("Saved model_ready.parquet, shape:", model_df.shape)
print(model_df.dtypes)

Saved model_ready.parquet, shape: (535920, 14)
qty                        float64
mass_kg                    float64
year                         int32
month                        int32
day                          int32
day_of_week                  int32
quarter                      int32
is_weekend                   int64
debtor_code                 object
product_code_grp            object
doc_type                    object
high_value                   int64
date                datetime64[ns]
value_zar                  float64
dtype: object
